# Step 2: The Baseline - Topological GCN

**Learning Objective:** Prove that 3D geometry (bond lengths) is essential for predicting quantum properties.

**Hypothesis:** *"A graph neural network that only knows bond existence (topology) but ignores bond length (geometry) will fail at quantum chemistry."*

---

## 1. Theoretical Background: Graph Convolutional Networks (GCN)

The **Graph Convolutional Network** (Kipf & Welling, ICLR 2017) is a fundamental GNN architecture for node classification and graph-level tasks.

### 1.1 GCN Layer Definition

For a graph $G = (V, E)$ with node features $\mathbf{H}^{(l)} \in \mathbb{R}^{N \times D}$ at layer $l$:

$$
\mathbf{H}^{(l+1)} = \sigma \left( \tilde{\mathbf{D}}^{-\frac{1}{2}} \tilde{\mathbf{A}} \tilde{\mathbf{D}}^{-\frac{1}{2}} \mathbf{H}^{(l)} \mathbf{W}^{(l)} \right)
$$

**Where:**
- $\tilde{\mathbf{A}} = \mathbf{A} + \mathbf{I}$ → Adjacency matrix with self-loops
- $\tilde{\mathbf{D}}$ → Degree matrix of $\tilde{\mathbf{A}}$: $\tilde{D}_{ii} = \sum_j \tilde{A}_{ij}$
- $\mathbf{W}^{(l)} \in \mathbb{R}^{D \times D'}$ → Learnable weight matrix
- $\sigma$ → Activation function (e.g., ReLU)

**Per-Node Form:**
$$
\mathbf{h}_i^{(l+1)} = \sigma \left( \sum_{j \in \mathcal{N}(i) \cup \{i\}} \frac{1}{\sqrt{\deg(i) \cdot \deg(j)}} \mathbf{W}^{(l)} \mathbf{h}_j^{(l)} \right)
$$

**Notation Mapping:**
- $\mathbf{h}_i^{(l)}$ → `node_feats[i]` (node feature vector at layer $l$)
- $\mathcal{N}(i)$ → Neighbors of node $i$ (from `edge_index`)
- $\deg(i)$ → Degree of node $i$ (number of neighbors)
- $\mathbf{W}^{(l)}$ → `gcn_layer.lin.weight` (learnable parameter)

---

### 1.2 What's Missing? GCN vs. MPNN

Recall the **General MPNN** message function from Step 1:
$$
m_v^{t+1} = \sum_{w \in \mathcal{N}(v)} M_t\left(h_v^t, h_w^t, \color{red}{e_{vw}}\right)
$$

Now compare to GCN:
$$
m_i^{(l+1)} = \sum_{j \in \mathcal{N}(i)} \frac{1}{\sqrt{\deg(i) \cdot \deg(j)}} \mathbf{W}^{(l)} \mathbf{h}_j^{(l)}
$$

**Critical Difference:**
- ❌ **GCN:** No $e_{vw}$ term → Edge features (bond distances) are **ignored**
- ✅ **MPNN:** $M_t(h_v, h_w, \color{red}{e_{vw}})$ → Messages are **conditioned on edge geometry**

**Consequence:** In GCN, a C-H bond at 1.09 Å and a stretched C-H bond at 2.0 Å are treated identically. This is catastrophic for quantum chemistry, where bond length determines energy.

---

### 1.3 Architecture for QM9

Our model:
1. **Embedding:** Convert atomic numbers (1-9) → dense vectors (e.g., 64-dim)
2. **GCN Layers:** 3 layers of graph convolutions (topology-only message passing)
3. **Global Pooling:** Aggregate node features → graph representation
   $$\mathbf{h}_{\text{graph}} = \frac{1}{|V|} \sum_{v \in V} \mathbf{h}_v^{(L)}$$
4. **Readout MLP:** Predict scalar target (e.g., HOMO-LUMO gap)

**Notation Mapping:**
- $\mathbf{h}_{\text{graph}}$ → `graph_embedding`
- $|V|$ → Number of atoms in molecule
- $\hat{y}$ → `prediction` (output of MLP)

---

In [ ]:
# Dependencies
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import QM9
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")

## 2. Dataset Preparation

We'll use the same QM9 dataset, but with a critical constraint:
- **Use:** `data.x` (atomic numbers) and `data.edge_index` (bond connectivity)
- **Ignore:** `data.pos` (coordinates) and `data.edge_attr` (bond features)

**Target:** HOMO-LUMO gap (`data.y[:, 4]`)

In [ ]:
# Load QM9
dataset = QM9(root='./data/QM9')

# Target index for HOMO-LUMO gap
TARGET_IDX = 4  # Index 4 in data.y corresponds to HOMO-LUMO gap

logger.info(f"Dataset size: {len(dataset)}")
logger.info(f"Target: HOMO-LUMO gap (index {TARGET_IDX})")

# Preprocessing: Extract only the target property
def preprocess_data(data: Data) -> Data:
    """
    Preprocess QM9 data for topology-only GCN.
    
    Args:
        data: Raw QM9 Data object
    
    Returns:
        data: Preprocessed Data with:
            - x: [N_atoms, 1] - Atomic numbers only (first feature)
            - edge_index: [2, N_edges] - Graph connectivity
            - y: [1] - Scalar target (HOMO-LUMO gap)
    
    Note:
        We deliberately IGNORE data.pos and data.edge_attr to test
        the hypothesis that topology alone is insufficient.
    """
    # Extract atomic numbers (first column of data.x)
    data.x = data.x[:, 0:1]  # [N_atoms, 1] - atomic number
    
    # Extract target (HOMO-LUMO gap)
    data.y = data.y[0, TARGET_IDX:TARGET_IDX+1]  # [1]
    
    return data


# Apply preprocessing
dataset_processed = [preprocess_data(data.clone()) for data in dataset]

# Train/Val/Test split (110k / 10k / ~14k)
train_dataset = dataset_processed[:110000]
val_dataset = dataset_processed[110000:120000]
test_dataset = dataset_processed[120000:]

logger.info(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# Inspect a batch
sample_batch = next(iter(train_loader))
logger.info(f"\nSample batch:")
logger.info(f"  data.x shape: {sample_batch.x.shape} → [Total_atoms_in_batch, 1]")
logger.info(f"  data.edge_index shape: {sample_batch.edge_index.shape}")
logger.info(f"  data.batch shape: {sample_batch.batch.shape} → Assigns each atom to a molecule")
logger.info(f"  data.y shape: {sample_batch.y.shape} → [Batch_size, 1]")
logger.info(f"  Number of molecules in batch: {sample_batch.y.shape[0]}")

## 3. Model Implementation: Topological GCN

**Architecture:**
```
Atomic Number → Embedding(64) → GCN(64) → GCN(64) → GCN(64) → GlobalMeanPool → MLP(128→64→1)
```

**Key Constraint:** No use of `data.pos` or `data.edge_attr` in the forward pass.

In [ ]:
class TopologyGCN(nn.Module):
    """
    Graph Convolutional Network that uses ONLY graph topology.
    
    This model deliberately ignores 3D coordinates and bond distances
    to demonstrate that geometric information is essential for
    quantum chemistry predictions.
    
    Architecture:
        1. Atom embedding (atomic number → 64-dim vector)
        2. 3 GCN layers (topology-based message passing)
        3. Global mean pooling (graph-level representation)
        4. MLP readout (regression to scalar target)
    
    Input:
        - data.x: [N_atoms, 1] - Atomic numbers
        - data.edge_index: [2, N_edges] - Graph connectivity
        - data.batch: [N_atoms] - Batch assignment
    
    Output:
        - prediction: [Batch_size, 1] - Predicted HOMO-LUMO gap
    """
    
    def __init__(
        self,
        num_atom_types: int = 10,  # Atomic numbers 1-9 + padding
        embedding_dim: int = 64,
        hidden_dim: int = 64,
        num_layers: int = 3,
        mlp_hidden_dim: int = 128
    ):
        super(TopologyGCN, self).__init__()
        
        # Atom embedding: atomic number → dense vector
        self.atom_embedding = nn.Embedding(num_atom_types, embedding_dim)
        
        # GCN layers (topology-only message passing)
        self.convs = nn.ModuleList()
        self.convs.append(GCNConv(embedding_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
        
        # Readout MLP
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, mlp_hidden_dim),
            nn.ReLU(),
            nn.Linear(mlp_hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, data: Data) -> torch.Tensor:
        """
        Forward pass.
        
        Args:
            data: PyG Data batch with x, edge_index, batch
        
        Returns:
            prediction: [Batch_size, 1] - Predicted property
        """
        # Extract inputs
        x = data.x.long().squeeze()  # [N_atoms] - atomic numbers
        edge_index = data.edge_index  # [2, N_edges]
        batch = data.batch  # [N_atoms]
        
        # Embedding: atomic number → feature vector
        h = self.atom_embedding(x)  # [N_atoms, embedding_dim]
        
        # GCN message passing (topology-only)
        for conv in self.convs:
            h = conv(h, edge_index)  # [N_atoms, hidden_dim]
            h = F.relu(h)
        
        # Global pooling: nodes → graph
        # h_graph = (1/|V|) * Σ h_v
        h_graph = global_mean_pool(h, batch)  # [Batch_size, hidden_dim]
        
        # Readout MLP: graph representation → scalar
        prediction = self.mlp(h_graph)  # [Batch_size, 1]
        
        return prediction


# Instantiate model
model = TopologyGCN(
    num_atom_types=10,
    embedding_dim=64,
    hidden_dim=64,
    num_layers=3,
    mlp_hidden_dim=128
).to(device)

logger.info(f"\nModel Architecture:")
logger.info(model)
logger.info(f"\nTotal parameters: {sum(p.numel() for p in model.parameters())}")

## 4. Training Loop

**Loss Function:** Mean Squared Error (MSE)
$$
\mathcal{L} = \frac{1}{B} \sum_{i=1}^{B} (\hat{y}_i - y_i)^2
$$

**Metrics:**
- **MAE** (Mean Absolute Error): $\frac{1}{B} \sum |\hat{y}_i - y_i|$
- **RMSE** (Root Mean Squared Error): $\sqrt{\text{MSE}}$

Units: eV (electron volts)

In [ ]:
def train_epoch(model: nn.Module, loader: DataLoader, optimizer, criterion) -> float:
    """
    Train for one epoch.
    
    Args:
        model: GCN model
        loader: Training data loader
        optimizer: PyTorch optimizer
        criterion: Loss function
    
    Returns:
        avg_loss: Average training loss (MSE)
    """
    model.train()
    total_loss = 0.0
    num_batches = 0
    
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        predictions = model(batch)  # [Batch_size, 1]
        targets = batch.y  # [Batch_size, 1]
        
        # Compute loss
        loss = criterion(predictions, targets)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
    
    return total_loss / num_batches


def evaluate(model: nn.Module, loader: DataLoader, criterion) -> Tuple[float, float, np.ndarray, np.ndarray]:
    """
    Evaluate model on a dataset.
    
    Args:
        model: GCN model
        loader: Data loader (validation or test)
        criterion: Loss function
    
    Returns:
        avg_loss: Average loss (MSE)
        mae: Mean Absolute Error
        predictions: All predictions (for visualization)
        targets: All ground truth values
    """
    model.eval()
    total_loss = 0.0
    total_mae = 0.0
    num_batches = 0
    
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            
            # Forward pass
            predictions = model(batch)
            targets = batch.y
            
            # Compute metrics
            loss = criterion(predictions, targets)
            mae = torch.abs(predictions - targets).mean()
            
            total_loss += loss.item()
            total_mae += mae.item()
            num_batches += 1
            
            # Store for visualization
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
    
    # Concatenate all predictions/targets
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    
    return total_loss / num_batches, total_mae / num_batches, all_predictions, all_targets


# Training setup
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50

# Training history
train_losses = []
val_losses = []
val_maes = []

logger.info(f"\n{'='*60}")
logger.info(f"Starting Training: Topological GCN (Baseline)")
logger.info(f"{'='*60}")

for epoch in range(num_epochs):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    train_losses.append(train_loss)
    
    # Validate
    val_loss, val_mae, _, _ = evaluate(model, val_loader, criterion)
    val_losses.append(val_loss)
    val_maes.append(val_mae)
    
    # Log progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        logger.info(
            f"Epoch [{epoch+1:3d}/{num_epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val MAE: {val_mae:.4f} eV"
        )

logger.info(f"\nTraining Complete!")
logger.info(f"Final Validation MAE: {val_maes[-1]:.4f} eV")
logger.info(f"Final Validation RMSE: {np.sqrt(val_losses[-1]):.4f} eV")

## 5. Visualization: Loss Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Training vs Validation Loss (MSE)
axes[0].plot(train_losses, label='Train Loss (MSE)', linewidth=2)
axes[0].plot(val_losses, label='Val Loss (MSE)', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (MSE, eV²)', fontsize=12)
axes[0].set_title('Training Curves - Topology-Only GCN', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Plot 2: Validation MAE
axes[1].plot(val_maes, label='Val MAE', color='orange', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('MAE (eV)', fontsize=12)
axes[1].set_title('Validation MAE - Topology-Only GCN', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

logger.info(f"\nObservation: Loss plateaus early → Model cannot learn meaningful patterns from topology alone.")

## 6. Visualization: Prediction vs. Ground Truth (The Failure Case)

In [ ]:
# Get predictions on test set
test_loss, test_mae, test_predictions, test_targets = evaluate(model, test_loader, criterion)

logger.info(f"\n{'='*60}")
logger.info(f"Test Set Performance")
logger.info(f"{'='*60}")
logger.info(f"Test MAE: {test_mae:.4f} eV")
logger.info(f"Test RMSE: {np.sqrt(test_loss):.4f} eV")

# Scatter plot: Predicted vs Ground Truth
fig, ax = plt.subplots(figsize=(8, 8))

# Plot predictions
ax.scatter(
    test_targets, 
    test_predictions, 
    alpha=0.3, 
    s=10, 
    c='blue', 
    label=f'GCN Predictions (MAE={test_mae:.4f} eV)'
)

# Plot perfect prediction line (y = x)
min_val = min(test_targets.min(), test_predictions.min())
max_val = max(test_targets.max(), test_predictions.max())
ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

ax.set_xlabel('Ground Truth HOMO-LUMO Gap (eV)', fontsize=13)
ax.set_ylabel('Predicted HOMO-LUMO Gap (eV)', fontsize=13)
ax.set_title(
    'Topology-Only GCN: Prediction vs Ground Truth\n(Test Set)', 
    fontsize=14, 
    fontweight='bold'
)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()

## 7. Analysis: Why Did the Model Fail?

### Expected Results
**Test MAE:** ~0.5-1.0 eV (very high for quantum chemistry standards)

### Root Cause Analysis

**1. Information Bottleneck**
- The model only knows:
  - Which atoms exist (C, H, N, O, F)
  - Which atoms are bonded (topology)
- The model does NOT know:
  - Bond lengths (1.0 Å vs 2.0 Å)
  - 3D geometry (cis vs trans isomers)
  - Spatial strain (steric hindrance)

**2. Identical Representation for Different Molecules**

Consider these two molecules:
- **Molecule A:** Ethane (C₂H₆) with C-C bond at 1.54 Å (relaxed)
- **Molecule B:** Ethane (C₂H₆) with C-C bond at 2.0 Å (stretched)

**Problem:** Both have identical topology → GCN produces identical predictions!

But in reality:
- Molecule A: Lower energy, normal HOMO-LUMO gap
- Molecule B: Higher energy, perturbed electronic structure

**3. The Scatter Plot Evidence**
- **High Dispersion:** Predictions are scattered far from the y=x line
- **No Strong Correlation:** The model has learned some weak signal (atom counts, graph size), but lacks the geometric information needed for accurate predictions

---

## 8. Conclusion

**Hypothesis Confirmed:** ✅
> *"A graph neural network that ignores 3D geometry cannot accurately predict quantum chemical properties."*

**Key Takeaway:**
$$
\text{Quantum Chemistry} = f(\text{Topology}, \color{red}{\text{Geometry}})
$$

**Missing Component:** Edge features $e_{vw}$ (bond distances)

**Mathematical Reminder:**
$$
\underbrace{m_v = \sum_{w \in \mathcal{N}(v)} M(h_v, h_w)}_{\text{GCN (Topology-Only)}} 
\quad \text{vs.} \quad
\underbrace{m_v = \sum_{w \in \mathcal{N}(v)} M(h_v, h_w, \color{red}{e_{vw}})}_{\text{MPNN (Geometry-Aware)}}
$$

---

## Next Step

**Step 3:** Implement the **Gilmer MPNN** with **Continuous Filter Convolution** (NNConv).
- Add edge features $e_{vw}$ (RBF-expanded distances)
- Use edge-conditioned message passing
- **Expected:** MAE drops from ~0.8 eV → ~0.05 eV (16× improvement!)

**Key Question:** How do we incorporate $e_{vw}$ into the message function?
**Answer (Preview):** Use an MLP to generate edge-specific weight matrices:
$$
M(h_v, h_w, e_{vw}) = \text{MLP}(e_{vw}) \cdot h_w
$$

See you in Step 3! 🚀